# Significância — BioBERTpt vs BERTimbau (RECLin-PT)

Compara os **dois modelos já treinados** no **mesmo** conjunto de teste e responde se a diferença é real ou ruído:

- **McNemar** (exato): os *padrões de erro* dos dois modelos diferem?
- **Bootstrap pareado** no F1 de `negation_of`: IC95 e p-valor da diferença na métrica-alvo.

**Não precisa de GPU nem Drive** — roda na CPU em segundos sobre as predições salvas.

### Pré-requisito
Rode **antes** os dois notebooks de treino (`baseline_biobertpt_colab.ipynb` e `baseline_bertimbau_colab.ipynb`) **até a célula de push** — eles publicam `results/baseline_*.preds.json` no GitHub, que este notebook baixa. Cadastre o secret `GITHUB_PAT` (🔑).

## 1. Clonar o repositório (traz as predições já publicadas)

In [ ]:
import os, subprocess
from google.colab import userdata

TOKEN = userdata.get('GITHUB_PAT')
REPO_NAME = 'RECLin-PT-Min'
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

url = f'https://{TOKEN}@github.com/angeloalsf/{REPO_NAME}'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)

subprocess.run(['git', '-C', REPO, 'config', 'user.name', GIT_USER_NAME], check=True)
subprocess.run(['git', '-C', REPO, 'config', 'user.email', GIT_USER_EMAIL], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', url], check=True)
print('Repo pronto em', REPO)


## 2. Dependências

`scikit-learn` e `scipy` já vêm no Colab; instalamos só por garantia.

In [ ]:
!pip install -q "scikit-learn>=1.4" "scipy>=1.10"


## 3. Conferir que as predições dos dois modelos existem

In [ ]:
a = os.path.join(REPO, 'results', 'baseline_biobertpt.preds.json')
b = os.path.join(REPO, 'results', 'baseline_bertimbau.preds.json')
for tag, p in [('BioBERTpt', a), ('BERTimbau', b)]:
    ok = os.path.isfile(p)
    print(('OK ' if ok else 'FALTA '), tag, '->', p)
assert os.path.isfile(a) and os.path.isfile(b), (
    'Faltam predições. Rode os dois notebooks de treino (até o push) e dê pull aqui.')


## 4. Rodar o teste de significância

`--target negation_of` é a classe-alvo do bootstrap; `--n-boot 10000` é o nº de reamostras. A ordem A/B não muda a conclusão (só o sinal da diferença).

In [ ]:
import sys, subprocess, json
out = 'results/significance_biobertpt_vs_bertimbau.json'
subprocess.run([sys.executable, 'src/significance.py',
                '--a', a, '--b', b,
                '--target', 'negation_of', '--n-boot', '10000', '--seed', '42',
                '--out', out], cwd=REPO, check=True)
rep = json.load(open(os.path.join(REPO, out), encoding='utf-8'))
print(json.dumps(rep, indent=2, ensure_ascii=False))


## 5. Leitura rápida do resultado

In [ ]:
bs = rep['paired_bootstrap']; mc = rep['mcnemar']
print(f"A = {rep['model_a']}")
print(f"B = {rep['model_b']}")
print(f"\nF1(negation_of): A={rep['target_f1']['a']:.4f}  B={rep['target_f1']['b']:.4f}  "
      f"A-B={rep['target_f1']['a_minus_b']:+.4f}")
print(f"Bootstrap: IC95=[{bs['ci95_low']:+.4f}, {bs['ci95_high']:+.4f}]  p={bs['p_value']:.4g}")
print(f"McNemar:   p={mc['p_value']:.4g}")
sig = bs['ci95_low'] > 0 or bs['ci95_high'] < 0
print('\n=> A diferença no F1 de negation_of', 'É' if sig else 'NÃO é',
      'estatisticamente significativa (IC95 ' + ('exclui' if sig else 'inclui') + ' zero).')


## 6. (Opcional) Salvar o relatório no GitHub

In [ ]:
subprocess.run(['git', '-C', REPO, 'add', '-f', out], check=True)
subprocess.run(['git', '-C', REPO, 'commit', '-m', 'significância: BioBERTpt vs BERTimbau'], check=False)
subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)
subprocess.run(['git', '-C', REPO, 'push'], check=False)
print('Push concluído (se houve mudanças).')
